# Flight Delay Forecasting — Final Project
## Dataset: US Flight Delay and Cancellation Data (2019–2023)
## Final Model: SARIMA(1,1,1)(1,0,1,12)
## Goal: Forecast monthly average arrival delays

## Problem Definition

Flight delays cost the US economy billions of dollars annually in lost productivity and operational costs. Airlines, airports, and travelers all suffer when delays go unpredicted and unmanaged. The COVID-19 pandemic created massive disruptions between 2020 and 2021, making historical delay patterns especially complex — delays turned negative as skies emptied, then surged past pre-pandemic levels as travel rebounded faster than the industry could staff up.

This project uses 3 million real domestic flight records to build a time series forecasting model that predicts monthly average arrival delays. By capturing trend and seasonal patterns in historical data, the model enables airlines, airports, and regulators to anticipate disruptions and allocate resources proactively.

## Pseudocode

```
ALGORITHM: Flight Delay Forecasting with SARIMA

INPUT: Raw flight data (3 million rows, 2019–2023)

STEP 1 — DATA CLEANING
   Remove cancelled flights (no delay to measure)
   Fill delay cause columns with 0 (meaningful zeros, not missing data)
   Drop CANCELLATION_CODE (97% empty after removing cancelled flights)
   Drop rows with missing ARR_DELAY (diverted flights)
   Drop 14 redundant or irrelevant columns
   Result: 2,913,802 rows, 17 clean columns

STEP 2 — BUILD TIME SERIES
   Convert FL_DATE to datetime format
   Group flights by month, calculate average ARR_DELAY per month
   Result: 56 monthly data points (Jan 2019 – Aug 2023)

STEP 3 — EXPLORATORY DATA ANALYSIS
   Time series plot with COVID structural break highlighted
   Average delay by airline (bar chart)
   Delay by month and year (heatmap)
   Seasonal decomposition (trend, seasonality, residual)

STEP 4 — STATIONARITY TESTING
   ADF test on raw series → p=0.084 (borderline non-stationary)
   KPSS test on raw series → p=0.0525 (borderline stationary)
   Both tests near boundary → COVID structural break is the likely cause
   Apply first differencing → ADF p=0.62 (still non-stationary)
   Apply second differencing → ADF p=0.0001 (stationary confirmed)
   Plot ACF → q=1 | Plot PACF → p=1

STEP 5 — TRAIN/TEST SPLIT
   Train: Jan 2019 – Dec 2022 (44 months)
   Test:  Jan 2023 – Aug 2023 (8 months)

STEP 6 — MODEL FITTING AND IMPROVEMENT
   Fit ARIMA(1,2,1) → unstable, forecast trends downward ❌
   Fit ARIMA(1,1,1) → stable forecast, d=1 sufficient despite ADF result ✅
   Fit SARIMA(1,1,1)(1,0,1,12) → adds seasonal component, further improves MAE ✅

STEP 7 — BASELINE AND BENCHMARKING
   Seasonal naive baseline: predict same month from prior year
   Compare ARIMA, SARIMA, Prophet (default), Prophet (tuned with COVID changepoints)

STEP 8 — EVALUATE
   MAE, RMSE, sMAPE on test set
   Final ranking: Seasonal Naive (2.96) < SARIMA (3.59) < ARIMA (3.89) < Prophet Tuned (4.54)

OUTPUT: Monthly flight delay forecast with error metrics and model comparison
```

## Algorithm Explanation: SARIMA

SARIMA stands for Seasonal AutoRegressive Integrated Moving Average. It extends ARIMA by adding a seasonal component, making it well-suited for time series that repeat patterns on a fixed cycle — like monthly flight delays peaking every June and July.

SARIMA is controlled by two sets of parameters: non-seasonal (p, d, q) and seasonal (P, D, Q, m).

**Non-Seasonal Component (p=1, d=1, q=1)**

- **AR (p=1):** Uses the previous month's delay to predict the next. "Last month's delay is a signal for this month's delay."
- **I (d=1):** One round of differencing removes the overall trend. We tested d=2 first (ARIMA(1,2,1)) but it over-differenced the series, causing the forecast to trend downward to -15 minutes — clearly wrong. d=1 produced a stable, realistic forecast.
- **MA (q=1):** Uses the previous month's forecast error to adjust the next prediction. "If we over-predicted last month, correct downward this month."

**Seasonal Component (P=1, D=0, Q=1, m=12)**

- **P=1:** Autoregressive term at the 12-month lag — last year's same month helps predict this year's.
- **D=0:** No seasonal differencing applied. With only 44 training months (fewer than 4 full seasons), seasonal differencing would remove too much information and destabilize the model.
- **Q=1:** Moving average correction at the 12-month lag.
- **m=12:** Seasonal period of 12 months, reflecting the annual cycle of summer peaks and fall/winter dips.

**Why SARIMA(1,1,1)(1,0,1,12) was chosen**

p and q were identified from ACF/PACF plots showing significant spikes at lag 1. d=1 was chosen based on forecast stability rather than the ADF test alone. The seasonal component was added after ARIMA was outperformed by a simple seasonal naive baseline — a clear signal that the model needed to explicitly account for the recurring annual pattern. SARIMA improved on ARIMA's MAE from 3.89 to 3.59 minutes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('/content/flights_sample_3m.csv.zip', compression='zip')
print(df.shape)
df.head()

## Data Loading

The dataset is the US Flight Delay and Cancellation Dataset (2019–2023), sourced from Kaggle and originally collected by the US Department of Transportation's Bureau of Transportation Statistics. It contains 3 million rows and 32 columns of real domestic flight records. Each row represents a single flight and includes the airline, origin, destination, scheduled and actual times, delay amounts by cause, and cancellation details.

This dataset was chosen because it is large enough to be realistic, messy enough to require meaningful cleaning, and spans five years — providing enough history for time series modeling while including the COVID structural break that makes the forecasting problem genuinely challenging.

In [ ]:
df.isnull().sum()

## Missing Value Analysis

Before cleaning, the dataset contains several columns with missing values that fall into three distinct categories:

**1. Cancelled and diverted flights** — Columns like DEP_TIME, ARR_TIME, and ARR_DELAY are missing for cancelled and diverted flights because those events never occurred. These rows will be removed entirely since there is no arrival delay to measure or predict.

**2. Cancellation code** — CANCELLATION_CODE has 2,920,860 missing values (97% of the dataset) because only cancelled flights have a cancellation reason. After removing cancelled flights this column becomes entirely empty and will be dropped.

**3. Delay cause columns** — DELAY_DUE_CARRIER, DELAY_DUE_WEATHER, DELAY_DUE_NAS, DELAY_DUE_SECURITY, and DELAY_DUE_LATE_AIRCRAFT each have millions of missing values, but these are not truly missing. The Bureau of Transportation Statistics only records a value when that delay type contributed to a late arrival. A blank means zero contribution — so these will be filled with 0, not dropped.

In [ ]:
df = df[df['CANCELLED'] == 0].copy()
print(f"Rows remaining: {len(df):,}")

## Dropping Cancelled Flights

Cancelled flights are removed because they have no arrival delay to measure — a flight that never departed cannot be late. Keeping them would introduce noise into the monthly averages that form our time series. All rows where CANCELLED == 1 are dropped, retaining only flights that actually flew.

In [ ]:
# Filling delay cause columns with 0 instead of dropping them because:
# DELAY_DUE_CARRIER     — blank means the airline caused NO delay, so 0 is correct
# DELAY_DUE_WEATHER     — blank means weather caused NO delay on that flight, so 0 is correct
# DELAY_DUE_NAS         — blank means air traffic control caused NO delay, so 0 is correct
# DELAY_DUE_SECURITY    — blank means there was NO security delay, so 0 is correct
# DELAY_DUE_LATE_AIRCRAFT — blank means the incoming aircraft was NOT late, so 0 is correct

delay_cols = ['DELAY_DUE_CARRIER', 'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS',
              'DELAY_DUE_SECURITY', 'DELAY_DUE_LATE_AIRCRAFT']
df[delay_cols] = df[delay_cols].fillna(0)

## Filling Delay Cause Columns with Zero

The five delay cause columns appear to have millions of missing values, but these are meaningful zeros — not missing data. The BTS only records a value when a delay type actually contributed minutes to a late arrival. If a flight arrived on time, or was delayed for a different reason, these fields are left blank rather than recorded as 0.

Filling with 0 is the correct approach because it accurately reflects what the data is saying: that delay type played no role in that particular flight. Dropping or imputing these rows statistically would misrepresent the data.

In [ ]:
df.isnull().sum()

In [ ]:
# 97% empty after removing cancelled flights — nothing left to keep
df = df.drop(columns=['CANCELLATION_CODE'])

## Dropping Cancellation Code

CANCELLATION_CODE records the reason a flight was cancelled (A = Airline, B = Weather, C = National Air System, D = Security). Since all cancelled flights were removed in the previous step, this column now contains only missing values and is dropped entirely.

In [ ]:
df = df.dropna(subset=['ARR_DELAY'])
print(f"Rows remaining: {len(df):,}")

## Dropping Rows with Missing Arrival Delay

After removing cancelled flights, a small number of rows still have a missing ARR_DELAY. These are diverted flights — redirected to a different airport due to weather or mechanical issues — that never completed their intended route. Since they have no meaningful arrival delay, they are dropped. This affects a very small fraction of the dataset and will not meaningfully impact monthly averages.

In [ ]:
df.isnull().sum()

In [ ]:
cols_to_drop = [
    'AIRLINE_DOT',       # duplicate of AIRLINE
    'DOT_CODE',          # internal govt ID, not useful for forecasting
    'FL_NUMBER',         # too granular for monthly forecasting
    'WHEELS_OFF',        # derived from dep time + taxi out
    'WHEELS_ON',         # derived from arr time + taxi in
    'TAXI_OUT',          # too granular for monthly forecasting
    'TAXI_IN',           # too granular
    'CRS_DEP_TIME',      # scheduled time not needed once we have delay amount
    'CRS_ARR_TIME',      # same reason
    'DEP_TIME',          # we care about delay amount, not actual clock time
    'ARR_TIME',          # same reason
    'ELAPSED_TIME',      # redundant with AIR_TIME
    'ORIGIN_CITY',       # redundant with ORIGIN airport code
    'DEST_CITY',         # redundant with DEST
]

df = df.drop(columns=cols_to_drop)
print(f"Columns remaining: {df.shape[1]}")
df.head()

## Dropping Redundant and Irrelevant Columns

The dataset is reduced from 32 columns to 17 by removing columns that are redundant, too granular, or irrelevant for monthly time series forecasting:

- **Duplicates:** AIRLINE_DOT, ORIGIN_CITY, DEST_CITY — same information as AIRLINE, ORIGIN, DEST in different formats
- **Internal identifiers:** DOT_CODE, FL_NUMBER — government tracking IDs with no predictive value
- **Derived columns:** WHEELS_OFF, WHEELS_ON, ELAPSED_TIME — calculated directly from other columns already in the dataset
- **Clock time columns:** DEP_TIME, ARR_TIME, CRS_DEP_TIME, CRS_ARR_TIME — since we are forecasting delay *amounts*, the actual clock times are unnecessary; DEP_DELAY and ARR_DELAY capture what we need
- **Too granular:** TAXI_OUT, TAXI_IN — meaningful at the individual flight level but not at monthly aggregation

The remaining 17 columns are a clean, focused dataset ready for time series analysis.

In [ ]:
df.to_csv('/content/df_clean.csv', index=False)
print("Saved!")

In [ ]:
from google.colab import files
files.download('/content/df_clean.csv')

Saving a copy of the cleaned dataset before moving into time series analysis.

In [ ]:
# Convert date column
df['FL_DATE'] = pd.to_datetime(df['FL_DATE'])

# Build monthly average arrival delay
monthly = df.groupby(df['FL_DATE'].dt.to_period('M'))['ARR_DELAY'].mean().round(2)
monthly.index = monthly.index.to_timestamp()

# Plot with COVID structural break highlighted
plt.figure(figsize=(12,5))
plt.plot(monthly)
plt.axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2021-06-01'),
            alpha=0.2, color='red', label='COVID Structural Break')
plt.title('Average Monthly Arrival Delay (2019-2023)')
plt.xlabel('Month')
plt.ylabel('Avg Delay (minutes)')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## Building and Plotting the Monthly Time Series

Individual flight records are aggregated into a single monthly average arrival delay, producing 56 data points from January 2019 to August 2023. The COVID structural break (March 2020 – June 2021) is shaded in red because it represents a regime change in the data — not ordinary noise or seasonality.

**Key patterns visible in the plot:**

- **2019:** Normal pre-pandemic operations, delays ranging 5–12 minutes
- **March–April 2020:** Sharp drop as COVID lockdowns emptied the skies; April 2020 hit -14 minutes, meaning flights were arriving *early* on average due to zero congestion
- **2021–2023:** Gradual recovery as travel rebounded, then consistent upward trend as demand surged faster than airlines could restore staffing
- **July 2023:** 17 minutes — the worst single month in the entire dataset

The COVID structural break is important for stationarity testing: the extreme dip creates apparent non-stationarity that is driven by a one-time shock rather than a true persistent trend. This distinction will directly affect how we choose the differencing parameter d.

In [ ]:
# Average arrival delay by airline
airline_delays = df.groupby('AIRLINE')['ARR_DELAY'].mean().round(2).sort_values(ascending=False)

plt.figure(figsize=(12,6))
plt.bar(airline_delays.index, airline_delays.values, color='steelblue')
plt.title('Average Arrival Delay by Airline (2019-2023)')
plt.xlabel('Airline')
plt.ylabel('Avg Delay (minutes)')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y')
plt.tight_layout()
plt.show()

## EDA: Average Arrival Delay by Airline

This bar chart breaks down average arrival delay by airline, sorted from worst to best, to understand whether the national average we are forecasting is driven by a few problem carriers or spread evenly across the industry.

Allegiant Air leads at 13.3 minutes, followed by JetBlue (12.4 min) and Frontier (11 min) — all low-cost carriers known for thin margins, tight turnarounds, and cascading operational delays. Delta and Endeavor Air perform best, with Endeavor showing negative average delays (flights arriving early on average). Endeavor operates exclusively for Delta, which runs one of the most reliable operations in the industry.

This chart provides useful context: the national monthly average we are forecasting is pulled upward by the worst-performing carriers, and a few airlines have an outsized effect on the overall number.

In [ ]:
import seaborn as sns

# Extract month and year
df['YEAR'] = df['FL_DATE'].dt.year
df['MONTH'] = df['FL_DATE'].dt.month

# Pivot table
heatmap_data = df.pivot_table(values='ARR_DELAY', index='YEAR', columns='MONTH', aggfunc='mean').round(2)
heatmap_data.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

plt.figure(figsize=(12,5))
sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='RdYlGn_r', center=0)
plt.title('Average Arrival Delay by Month and Year (2019-2023)')
plt.xlabel('Month')
plt.ylabel('Year')
plt.tight_layout()
plt.show()

# Color guide
print("🟢 Green = low or negative delays (flights arriving early)")
print("🔴 Red/Orange = high delays")
print("🟡 Yellow = near on-time")

## EDA: Heatmap — Delay by Month and Year

This heatmap provides a two-dimensional view of delays across both month and year simultaneously, making seasonal and year-over-year patterns easy to spot at a glance.

**Key observations:**

- **2020 (full row green):** The entire year shows negative or near-zero delays — COVID eliminated congestion entirely. April 2020 at -13.9 min is the lowest point in the dataset.
- **June/July (consistently red every year):** Summer travel demand peaks and overwhelms airline capacity reliably every year. July 2023 (17.0 min) is the worst single month in the dataset. This consistent seasonal pattern is exactly what SARIMA is designed to model.
- **December 2022 (13.4 min):** The Southwest Airlines holiday meltdown, where thousands of flights were cancelled and delayed, pulled the entire industry's monthly average up significantly.
- **2021–2023 (generally orange/red):** Post-pandemic recovery delays are consistently elevated as demand surged faster than airlines could rehire and retrain staff.

The strong and consistent June/July seasonal pattern visible here is the primary motivation for upgrading from ARIMA to SARIMA.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

decomposition = seasonal_decompose(monthly, model='additive', period=12)

fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(12, 10))

decomposition.observed.plot(ax=ax1)
ax1.set_title('Observed')
ax1.set_ylabel('Delay (min)')

decomposition.trend.plot(ax=ax2)
ax2.set_title('Trend')
ax2.set_ylabel('Delay (min)')

decomposition.seasonal.plot(ax=ax3)
ax3.set_title('Seasonality')
ax3.set_ylabel('Delay (min)')

decomposition.resid.plot(ax=ax4)
ax4.set_title('Residual')
ax4.set_ylabel('Delay (min)')

plt.tight_layout()
plt.show()

## EDA: Seasonal Decomposition

Seasonal decomposition breaks the time series into three components — trend, seasonality, and residual — to understand what is driving the patterns we observe. An additive model is used (components sum to the observed value) with a period of 12 since our data is monthly.

- **Trend:** Isolates the long-term direction. Captures the full COVID story — declining from ~5 min in 2019 to around -5 min at the pandemic's peak, then recovering and climbing back above pre-pandemic levels by 2023.
- **Seasonality:** Confirms the repeating annual cycle, with peaks around +4 min in summer and dips around -2 min in fall and winter — consistent with the heatmap patterns.
- **Residual:** What remains after removing trend and seasonality. Residuals fluctuate between -5 and +5 min with no clear pattern, indicating the decomposition has captured most of the meaningful variation. The remaining noise is largely unpredictable without additional data sources like weather or staffing levels.

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss

# ADF Test — null hypothesis: series IS non-stationary
result = adfuller(monthly)
print('=== Augmented Dickey-Fuller (ADF) Test ===')
print(f'ADF Statistic: {round(result[0], 4)}')
print(f'p-value:       {round(result[1], 4)}')
print('Critical Values:')
for key, value in result[4].items():
    print(f'   {key}: {round(value, 4)}')
if result[1] <= 0.05:
    print('\n✅ ADF: Series is STATIONARY (p <= 0.05)')
else:
    print('\n❌ ADF: Series is NON-STATIONARY (p > 0.05)')

print()

# KPSS Test — null hypothesis: series IS stationary
kpss_stat, kpss_p, lags, crit = kpss(monthly, regression='c')
print('=== KPSS Test ===')
print(f'KPSS Statistic: {kpss_stat:.4f}')
print(f'p-value:        {kpss_p:.4f}')
if kpss_p <= 0.05:
    print('\n❌ KPSS: Series is NON-STATIONARY (p <= 0.05)')
else:
    print('\n✅ KPSS: Series is STATIONARY (p > 0.05)')

## Stationarity Testing — ADF and KPSS

Before fitting any time series model, the series must be stationary — meaning it has a consistent mean and variance over time with no persistent trend. ARIMA and SARIMA both assume stationarity, so this step is essential.

Two complementary tests are used together because their null hypotheses are opposite, making them a reliable pair:

- **ADF (p=0.084):** Fails to reject non-stationarity at the 5% threshold — suggests the series may be non-stationary. But 0.084 is borderline, not a clear rejection.
- **KPSS (p=0.0525):** Barely fails to reject stationarity — suggests the series may actually be stationary.

**Interpretation:** Both tests land right on the boundary. This is a strong signal that the series is not *strongly* non-stationary — the COVID structural break (a one-time regime change) is likely distorting both tests to look worse than the underlying series actually is. True non-stationarity from a persistent trend would produce much more decisive results.

This explains why d=2 over-differenced the data and why d=1 ultimately produces a better forecast — there was not enough genuine non-stationarity to require two rounds of differencing.

In [ ]:
# Apply first differencing
monthly_diff = monthly.diff().dropna()

plt.figure(figsize=(12,5))
plt.plot(monthly_diff)
plt.title('First-Differenced Monthly Arrival Delay')
plt.xlabel('Month')
plt.ylabel('Change in Avg Delay (minutes)')
plt.grid(True)
plt.show()

# ADF on differenced series
result2 = adfuller(monthly_diff)
print(f'ADF Statistic: {round(result2[0], 4)}')
print(f'p-value:       {round(result2[1], 4)}')
if result2[1] <= 0.05:
    print('\n✅ Series is STATIONARY after first differencing!')
else:
    print('\n❌ Still non-stationary — applying second differencing')

## First Differencing

Differencing removes trend by replacing each value with the change from the previous month. Instead of asking "what was the average delay in March?", we ask "how much did delays change from February to March?".

After first differencing, the ADF test returns p=0.619 — still well above 0.05, suggesting the series remains non-stationary. The differenced plot also shows irregular swings rather than a consistent fluctuation around zero, likely due to the COVID period creating a shock too large to remove with a single round of differencing.

A second round of differencing will be applied. However, as we will see during model fitting, the ADF result should not be the only guide — the actual forecast output is equally important for validating the choice of d.

In [ ]:
# Apply second differencing
monthly_diff2 = monthly_diff.diff().dropna()

plt.figure(figsize=(12,5))
plt.plot(monthly_diff2)
plt.title('Second-Differenced Monthly Arrival Delay')
plt.xlabel('Month')
plt.ylabel('Change in Avg Delay (minutes)')
plt.grid(True)
plt.show()

# ADF on second-differenced series
result3 = adfuller(monthly_diff2)
print(f'ADF Statistic: {round(result3[0], 4)}')
print(f'p-value:       {round(result3[1], 4)}')
if result3[1] <= 0.05:
    print('\n✅ Series is STATIONARY after second differencing!')
else:
    print('\n❌ Still non-stationary')

## Second Differencing

Second differencing looks at the change in the month-to-month changes — removing both trend and any remaining acceleration in the data.

After second differencing, the ADF test returns p=0.0001 — far below the 0.05 threshold and even below the 1% critical value. The series is now statistically stationary. The plot confirms this visually: the series fluctuates consistently around zero with no visible drift.

This suggests using d=2 in the ARIMA model. However, the combined ADF/KPSS results from the previous section — both sitting near the boundary — warned us that the series may not truly need this much differencing. The next section will show that ARIMA(1,2,1) over-differenced the series and produced an unstable forecast, while ARIMA(1,1,1) with d=1 is stable and accurate. **The lesson: always validate the differencing choice by inspecting the actual forecast, not just the statistical tests.**

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

plot_acf(monthly_diff2, lags=20, ax=ax1)
ax1.set_title('Autocorrelation Function (ACF) — used to identify q')

plot_pacf(monthly_diff2, lags=20, ax=ax2)
ax2.set_title('Partial Autocorrelation Function (PACF) — used to identify p')

plt.tight_layout()
plt.show()

## ACF and PACF — Identifying p and q

With the series stationary, ACF and PACF plots identify the remaining ARIMA parameters p and q. Any spike extending beyond the blue shaded confidence region is statistically significant.

- **ACF (finds q):** Shows a significant spike at lag 1 that cuts off sharply → **q=1**
- **PACF (finds p):** Shows a significant spike at lag 1 that cuts off sharply → **p=1**

Combined with d=1 from the differencing analysis, the starting model is **ARIMA(1,1,1)**. These same p and q values carry over to the SARIMA model.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

# Train: Jan 2019 – Dec 2022 (44 months)
# Test:  Jan 2023 – Aug 2023 (8 months)
train = monthly[:-12]
test = monthly[-12:]

# Fit ARIMA(1,2,1) first — based on ADF suggesting d=2
model = ARIMA(train, order=(1, 2, 1))
model_fit = model.fit()
print(model_fit.summary())

## Train/Test Split and ARIMA(1,2,1) — First Attempt

The data is split into 44 training months (Jan 2019 – Dec 2022) and 8 test months (Jan 2023 – Aug 2023). The model is trained only on historical data and forecasts the test period — simulating real-world deployment where future values are unknown.

ARIMA(1,2,1) is fitted first, based on the ADF test suggesting d=2. The model summary shows reasonable diagnostic statistics (Ljung-Box p=0.74, confirming residuals are random noise), but as the next plot reveals, the actual forecast is severely wrong.

In [ ]:
# Forecast 12 months
forecast = model_fit.forecast(steps=12)

plt.figure(figsize=(12,5))
plt.plot(train, label='Training Data')
plt.plot(test, label='Actual', color='green')
plt.plot(test.index, forecast, label='Forecast ARIMA(1,2,1)', color='red', linestyle='--')
plt.title('ARIMA(1,2,1) Forecast vs Actual — Failed Attempt')
plt.xlabel('Month')
plt.ylabel('Avg Delay (minutes)')
plt.legend()
plt.grid(True)
plt.show()

## ARIMA(1,2,1) — Failed Forecast

The ARIMA(1,2,1) forecast trends sharply downward from ~5 minutes in January 2023 to approximately -15 minutes by August 2023 — while actual delays ranged between 2 and 17 minutes. This is clearly wrong.

This is a textbook case of **over-differencing**. When d=2 is applied to a series that does not truly require it, the model removes too much information and loses track of the actual level of the data, causing the forecast to drift in the wrong direction.

This failed forecast demonstrates something important: **achieving statistical stationarity through differencing does not guarantee a good forecast.** The choice of d must be validated through the actual forecast output. The ADF/KPSS boundary results from earlier foreshadowed exactly this problem — the series was never strongly non-stationary to begin with.

In [ ]:
# Reduce d from 2 to 1 — more appropriate given borderline stationarity results
model2 = ARIMA(train, order=(1, 1, 1))
model_fit2 = model2.fit()

forecast2 = model_fit2.forecast(steps=12)

plt.figure(figsize=(12,5))
plt.plot(train, label='Training Data')
plt.plot(test, label='Actual', color='green')
plt.plot(test.index, forecast2, label='Forecast ARIMA(1,1,1)', color='red', linestyle='--')
plt.title('ARIMA(1,1,1) Forecast vs Actual — Improved')
plt.xlabel('Month')
plt.ylabel('Avg Delay (minutes)')
plt.legend()
plt.grid(True)
plt.show()

## ARIMA(1,1,1) — Improved Forecast

Reducing d from 2 to 1 produces a dramatically better result. The forecast stays in a realistic 5–8 minute range throughout the test period and correctly identifies the general level of delays in 2023 without drifting off course.

The model misses the sharp July 2023 spike to 17 minutes — but this is expected behavior. ARIMA captures trend and general patterns, not individual month-to-month volatility. That spike would require external data (weather events, staffing levels) to predict. For national monthly forecasting, getting the right range and direction is the primary goal.

This confirms that d=1 is the correct choice for this dataset, and the ARIMA(1,1,1) model serves as the solid baseline for further improvement.

In [ ]:
# Forecast with 95% confidence interval
forecast_obj = model_fit2.get_forecast(steps=12)
conf_int = forecast_obj.conf_int()

plt.figure(figsize=(12,5))
plt.plot(train, label='Training Data')
plt.plot(test, label='Actual', color='green')
plt.plot(test.index, forecast2, label='ARIMA(1,1,1) Forecast', color='red', linestyle='--')
plt.fill_between(test.index,
                 conf_int.iloc[:,0],
                 conf_int.iloc[:,1],
                 color='red', alpha=0.1, label='95% Confidence Interval')
plt.title('ARIMA(1,1,1) Forecast with 95% Confidence Interval')
plt.xlabel('Month')
plt.ylabel('Avg Delay (minutes)')
plt.legend()
plt.grid(True)
plt.show()

## ARIMA(1,1,1) — Forecast with Confidence Interval

The 95% confidence interval (pink shading) communicates uncertainty around each predicted value — the model is 95% confident the true value falls within the shaded region.

- **Widening interval:** The band starts narrow in January 2023 and gradually expands. This is correct and expected behavior — uncertainty grows the further out we forecast.
- **Most actuals within the interval:** The majority of green data points fall inside the shaded region, indicating the model is well-calibrated.
- **July 2023 exception:** The 17-minute spike falls outside the interval, reflecting an unusually severe month that historical patterns alone could not anticipate.

This plot confirms ARIMA(1,1,1) is stable and appropriately uncertain. The next step is to evaluate it quantitatively and then extend it to SARIMA to capture the seasonal pattern.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae = mean_absolute_error(test, forecast2)
rmse = np.sqrt(mean_squared_error(test, forecast2))
mape = (abs((test.values - forecast2.values) / test.values).mean()) * 100

# sMAPE — more robust when actuals are near zero or negative
smape = (2 * abs(test.values - forecast2.values) /
         (abs(test.values) + abs(forecast2.values))).mean() * 100

print('=== ARIMA(1,1,1) Evaluation Metrics ===')
print(f'MAE:   {mae:.2f} minutes')
print(f'RMSE:  {rmse:.2f} minutes')
print(f'MAPE:  {mape:.2f}%  ← unreliable (see note below)')
print(f'sMAPE: {smape:.2f}%')

## ARIMA(1,1,1) — Evaluation Metrics

- **MAE (3.89 min):** The model's predictions are off by ~4 minutes on average. For a national monthly delay forecast, this is a practically useful result.
- **RMSE (4.81 min):** Slightly higher than MAE, indicating a few months have larger-than-average errors — most likely July 2023. The small MAE/RMSE gap suggests errors are fairly consistent with no single outlier dominating.
- **MAPE (81%):** Looks alarming but is misleading. MAPE divides by the actual value — when actuals are near zero or negative (as during COVID), the denominator blows up and inflates the percentage dramatically. This metric should not be used to evaluate this dataset.
- **sMAPE (53.70%):** More robust than MAPE since it uses the average of actual and predicted in the denominator. Still elevated due to COVID-period distortion, but a more honest representation than MAPE. MAE and RMSE remain the primary metrics.

In [ ]:
# Seasonal naive: predict same month from prior year
seasonal_naive = [monthly[monthly.index.month == m].iloc[-2]
                  for m in test.index.month]
mae_naive = mean_absolute_error(test, seasonal_naive)

print('=== Baseline Comparison ===')
print(f'Seasonal Naive MAE: {mae_naive:.2f} minutes')
print(f'ARIMA(1,1,1) MAE:   {mae:.2f} minutes')

## Seasonal Naive Baseline

Before comparing models, a seasonal naive baseline is established. This model makes no statistical assumptions — it simply predicts that next July will look like last July, next January will look like last January, and so on.

**Seasonal Naive MAE: 2.96 min — outperforms ARIMA (3.89 min).**

This is an important and honest finding. When a naive baseline beats your model, it means the model is not fully capturing the seasonal pattern. Rather than hiding this result, it directly motivates the next step: upgrading to SARIMA, which explicitly models the 12-month seasonal cycle instead of ignoring it.

A model that beats a naive baseline has proven it adds real forecasting value. ARIMA does not — yet.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Use 8 test months to match the actual test period (Jan–Aug 2023)
test_8 = test.iloc[:8]

# SARIMA(1,1,1)(1,0,1,12)
# Non-seasonal: same as ARIMA(1,1,1)
# Seasonal: P=1, D=0 (no seasonal differencing — too few seasons), Q=1, m=12
sarima_model = SARIMAX(train, order=(1,1,1), seasonal_order=(1,0,1,12))
sarima_fit = sarima_model.fit(disp=False)

# Forecast 8 steps
sarima_forecast = sarima_fit.forecast(steps=8)

# Metrics
mae_sarima = mean_absolute_error(test_8, sarima_forecast)
rmse_sarima = np.sqrt(mean_squared_error(test_8, sarima_forecast))

print('=== SARIMA(1,1,1)(1,0,1,12) Results ===')
print(f'SARIMA MAE:         {mae_sarima:.2f} minutes')
print(f'ARIMA MAE:          {mae:.2f} minutes')
print(f'Seasonal Naive MAE: {mae_naive:.2f} minutes')

In [ ]:
# Plot SARIMA vs ARIMA vs Actual
plt.figure(figsize=(12,5))
plt.plot(train, label='Training Data')
plt.plot(test_8, label='Actual', color='green')
plt.plot(test_8.index, sarima_forecast, label='SARIMA(1,1,1)(1,0,1,12)', color='purple', linestyle='--')
plt.plot(test_8.index, forecast2.iloc[:8], label='ARIMA(1,1,1)', color='red', linestyle='--')
plt.title('SARIMA vs ARIMA Forecast Comparison')
plt.xlabel('Month')
plt.ylabel('Avg Delay (minutes)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## SARIMA(1,1,1)(1,0,1,12) — Seasonal Model

SARIMA extends ARIMA by adding seasonal autoregressive and moving average terms at the 12-month lag. This means the model can now learn that July 2023 should look somewhat like July 2022, rather than relying entirely on recent month-to-month patterns.

**Why D=0 (no seasonal differencing):** With only 44 training months, there are fewer than 4 complete 12-month seasons in the training data. Applying seasonal differencing would remove critical information the model needs to learn seasonal patterns, causing instability — similar to how d=2 over-differenced the non-seasonal component.

**Results:**
- SARIMA MAE: **3.59 min** — improved over ARIMA (3.89 min) by 0.30 minutes
- Still behind the seasonal naive baseline (2.96 min), but the gap has narrowed

The improvement confirms that explicitly modeling the seasonal component adds value. The naive baseline remains ahead primarily because of data volume constraints — with fewer than 4 full training seasons, SARIMA cannot fully learn the seasonal pattern. A longer dataset (5–10 years) would likely allow SARIMA to close or surpass the naive baseline.

In [ ]:
from prophet import Prophet

# Prepare data in Prophet format
prophet_df = pd.DataFrame({'ds': train.index, 'y': train.values})

# Default Prophet model
prophet_model = Prophet(yearly_seasonality=True)
prophet_model.fit(prophet_df)

future = prophet_model.make_future_dataframe(periods=12, freq='MS')
prophet_forecast = prophet_model.predict(future)

plt.figure(figsize=(12,5))
plt.plot(train, label='Training Data')
plt.plot(test, label='Actual', color='green')
plt.plot(prophet_forecast['ds'].tail(12),
         prophet_forecast['yhat'].tail(12),
         label='Prophet Forecast (default)', color='orange', linestyle='--')
plt.title('Prophet Forecast vs Actual')
plt.xlabel('Month')
plt.ylabel('Avg Delay (minutes)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Tuned Prophet — COVID changepoints explicitly specified
prophet_model2 = Prophet(
    yearly_seasonality=True,
    changepoint_prior_scale=0.5,          # more flexible trend
    changepoints=['2020-03-01', '2021-06-01']  # COVID structural breaks
)
prophet_model2.fit(prophet_df)

future2 = prophet_model2.make_future_dataframe(periods=12, freq='MS')
prophet_forecast2 = prophet_model2.predict(future2)

prophet_test_forecast2 = prophet_forecast2['yhat'].tail(12).values

mae_prophet2 = mean_absolute_error(test, prophet_test_forecast2)
rmse_prophet2 = np.sqrt(mean_squared_error(test, prophet_test_forecast2))

print('=== Tuned Prophet Results ===')
print(f'Tuned Prophet MAE:  {mae_prophet2:.2f} minutes')
print(f'Tuned Prophet RMSE: {rmse_prophet2:.2f} minutes')
print(f'ARIMA MAE:          {mae:.2f} minutes')
print(f'ARIMA RMSE:         {rmse:.2f} minutes')

## Facebook Prophet — Default and Tuned

Prophet is an open-source forecasting library developed by Meta, designed to handle trend changes, seasonality, and holidays automatically without requiring manual stationarity testing. It is a widely used alternative to ARIMA in industry and serves as a strong benchmark here.

Two versions are tested on the same training data:

**Default Prophet (yearly_seasonality=True):** Fits automatically and forecasts the test period. Stays in a realistic range but shows more volatility than ARIMA and tracks the actual values less consistently.

**Tuned Prophet:** COVID changepoints are manually specified at March 2020 and June 2021, and the changepoint flexibility is increased (prior_scale=0.5) to better handle the post-pandemic recovery. This represents a fair comparison — Prophet is given explicit knowledge of the structural break that ARIMA discovered through its own differencing process.

Despite these advantages, the tuned Prophet (MAE: 4.54) performed *worse* than both the default Prophet and ARIMA. This is consistent with Prophet's known behavior on short time series — with only 56 monthly points, there is not enough history for Prophet to fully leverage its flexibility. ARIMA and SARIMA are better suited to datasets of this length.

In [ ]:
prophet_test_forecast = prophet_forecast['yhat'].tail(12).values
mae_prophet = mean_absolute_error(test, prophet_test_forecast)
rmse_prophet = np.sqrt(mean_squared_error(test, prophet_test_forecast))

print('=== Final Model Comparison ===')
print(f'{"Model":<35} {"MAE":>10} {"RMSE":>10}')
print('-' * 57)
print(f'{"Seasonal Naive":<35} {mae_naive:>10.2f} {"—":>10}')
print(f'{"SARIMA(1,1,1)(1,0,1,12)":<35} {mae_sarima:>10.2f} {rmse_sarima:>10.2f}')
print(f'{"ARIMA(1,1,1)":<35} {mae:>10.2f} {rmse:>10.2f}')
print(f'{"Prophet (default)":<35} {mae_prophet:>10.2f} {rmse_prophet:>10.2f}')
print(f'{"Prophet (tuned + COVID changepoints)":<35} {mae_prophet2:>10.2f} {rmse_prophet2:>10.2f}')

In [ ]:
# Visual comparison — all models
plt.figure(figsize=(12,5))
plt.plot(train, label='Training Data', color='steelblue')
plt.plot(test_8, label='Actual', color='green')
plt.plot(test_8.index, sarima_forecast, label='SARIMA(1,1,1)(1,0,1,12)', color='purple', linestyle='--')
plt.plot(test_8.index, forecast2.iloc[:8], label='ARIMA(1,1,1)', color='red', linestyle='--')
plt.plot(test_8.index, prophet_test_forecast[:8], label='Prophet (default)', color='orange', linestyle='--')
plt.title('All Models vs Actual — Final Comparison')
plt.xlabel('Month')
plt.ylabel('Avg Delay (minutes)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Final Model Comparison and Conclusions

| Model | MAE | RMSE |
|---|---|---|
| Seasonal Naive | 2.96 min | — |
| **SARIMA(1,1,1)(1,0,1,12)** | **3.59 min** | **4.48 min** |
| ARIMA(1,1,1) | 3.89 min | 4.81 min |
| Prophet (default) | 4.28 min | 5.29 min |
| Prophet (tuned) | 4.54 min | 5.50 min |

**SARIMA(1,1,1)(1,0,1,12) is selected as the final model.** It outperforms plain ARIMA by 0.30 minutes MAE and beats both Prophet versions by a significant margin, demonstrating that explicitly modeling the 12-month seasonal cycle adds real forecasting value.

The seasonal naive baseline (2.96 min) still leads overall. This is expected and explainable: with fewer than 4 complete training seasons, SARIMA cannot fully learn the seasonal pattern from limited data. The naive model has perfect seasonal knowledge baked in by design. A longer historical dataset (5–10 years) would likely allow SARIMA to close or surpass the naive baseline.

**Key lessons from this project:**

1. Statistical tests alone do not determine model parameters — ARIMA(1,2,1) passed stationarity tests but failed in practice. Always validate with the actual forecast.
2. When a naive baseline beats your model, it is a signal to add seasonality — not a failure.
3. Prophet is powerful but requires longer time series; SARIMA is better suited to monthly data with limited history.
4. The COVID structural break is best treated as a known regime change, not ordinary noise — and explicitly acknowledging it strengthens both the analysis and the stationarity argument.

**Future improvements:** Adding external features (weather severity index, TSA throughput, fuel prices, staffing levels) as exogenous regressors in a SARIMAX model would capture the month-to-month volatility — like the July 2023 spike to 17 minutes — that trend and seasonality alone cannot predict.